# Version 8.1 : Ensemble RSF + XGBoost Optimisé

**Stratégie d'ensemble optimisée** :
- 🌲 **RSF** : 90 features sélectionnées (V4, C-index: 0.7404)
- 🚀 **XGBoost** : Features optimisées + hyperparams de V6.1 (C-index: 0.741)
- 🎯 **Optuna** : Optimisation des poids d'ensemble

**Diversité maximale** :
- RSF : Feature set optimisé pour Random Forest
- XGBoost : Feature selection + tout hyperparams optimaux

**Objectif** : C-index > 0.76

## 1. Setup

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported")

✓ Libraries imported


## 2. Configuration

In [20]:
# RSF: 90 best features (from V4)
RSF_FEATURES = [
    'BM_BLAST', 'HB', 'PLT', 'vaf_sum', 'mutation_count_total', 'effect_non_synonymous_codon',
    'gene_RUNX1_count', 'WBC', 'cyto_total_anomalies', 'gene_RUNX1_present', 'ANC', 'cyto_normal',
    'MONOCYTES', 'vaf_mean', 'gene_TP53_count', 'cyto_complex', 'gene_TP53_present', 'cyto_loss_count',
    'effect_frameshift_variant', 'gene_NRAS_present', 'gene_NRAS_count', 'CENTER_PV', 'effect_PTD',
    'gene_ASXL1_count', 'cyto_chr7_affected', 'gene_TET2_count', 'gene_SF3B1_present', 'vaf_max',
    'gene_DNMT3A_present', 'gene_U2AF1_present', 'gene_SF3B1_count', 'CENTER_RMCN', 'cyto_chr18_affected',
    'cyto_gain_count', 'gene_U2AF1_count', 'cyto_chr1_affected', 'cyto_chrX_affected', 'cyto_other_count',
    'cyto_transloc_count', 'cyto_chr3_affected', 'gene_NF1_count', 'cyto_chr7_abnormal', 'gene_EZH2_present',
    'cyto_chr6_affected', 'CENTER_TUD', 'gene_SRSF2_count', 'gene_DNMT3A_count', 'gene_PHF6_count',
    'gene_TET2_present', 'CENTER_ROM', 'effect_inframe_codon_gain', 'gene_BCOR_count', 'CENTER_GESMD',
    'gene_CUX1_present', 'cyto_chr4_affected', 'gene_ZRSR2_count', 'gene_ZRSR2_present', 'CENTER_DUS',
    'cyto_chrY_affected', 'CENTER_CGM', 'cyto_del5q', 'cyto_chr15_affected', 'cyto_chr14_affected',
    'gene_NF1_present', 'CENTER_DUTH', 'CENTER_HIAE', 'effect_complex_change_in_transcript',
    'effect_initiator_codon_change', 'cyto_chr3_abnormal', 'cyto_chr19_affected', 'cyto_chr10_affected',
    'CENTER_UMG', 'CENTER_REL', 'CENTER_IHBT', 'CENTER_HMS', 'CENTER_MSK', 'effect_3_prime_UTR_variant',
    'effect_2KB_upstream_variant', 'effect_ITD', 'effect_stop_lost', 'cyto_inv_count',
    'effect_stop_retained_variant', 'effect_inframe_variant', 'effect_synonymous_codon', 'CENTER_VU',
    'CENTER_UOXF', 'CENTER_UOB', 'gene_KRAS_present', 'gene_KRAS_count', 'cyto_has_inv'
]

# RSF hyperparameters (from V4)
RSF_PARAMS = {
    'n_estimators': 350,
    'min_samples_split': 27,
    'min_samples_leaf': 5,
    'max_features': 0.2,
    'max_depth': 23,
    'bootstrap': True,
    'n_jobs': -1,
    'random_state': 42,
    'verbose': 0
}


# XGBoost parameters (OPTIMIZED from V6.1 - C-index: 0.7413)
XGB_PARAMS_BASE = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'tree_method': 'hist',
    'learning_rate': 0.0744,
    'max_depth': 3,
    'min_child_weight': 9,
    'subsample': 0.6194,
    'colsample_bytree': 0.8872,
    'reg_alpha': 0.0124,
    'reg_lambda': 0.0043,
    'gamma': 0.0600,
}

print("✓ Configuration loaded")
print(f"  RSF: {len(RSF_FEATURES)} features")
print(f"  XGBoost: ALL base + advanced features")

✓ Configuration loaded
  RSF: 90 features
  XGBoost: ALL base + advanced features


## 3. Data Loading & Feature Engineering

In [13]:
# Same pipeline as V8
DATA_PATH = r"C:\Users\guill\Desktop\Data Challenge QRT\Data-Challenge-Prediction-de-Survie"

clinical_train = pd.read_csv(f"{DATA_PATH}\\X_train\\clinical_train.csv")
target_train = pd.read_csv(f"{DATA_PATH}\\target_train.csv")
clinical_test = pd.read_csv(f"{DATA_PATH}\\X_test\\clinical_test.csv")
molecular_train = pd.read_csv(f"{DATA_PATH}\\X_train\\molecular_train.csv")
molecular_test = pd.read_csv(f"{DATA_PATH}\\X_test\\molecular_test.csv")

print("✓ Data loaded")

✓ Data loaded


In [14]:
# Feature engineering (condensed)
def create_cytogenetic_features(clinical_df):
    cyto_features = pd.DataFrame(index=clinical_df['ID'])
    cyto_col = clinical_df.set_index('ID')['CYTOGENETICS'].fillna('')
    cyto_features['cyto_del_count'] = cyto_col.str.count(r'del\(')
    cyto_features['cyto_has_del'] = (cyto_features['cyto_del_count'] > 0).astype(int)
    cyto_features['cyto_transloc_count'] = cyto_col.str.count(r't\(')
    cyto_features['cyto_has_transloc'] = (cyto_features['cyto_transloc_count'] > 0).astype(int)
    cyto_features['cyto_inv_count'] = cyto_col.str.count(r'inv\(')
    cyto_features['cyto_has_inv'] = (cyto_features['cyto_inv_count'] > 0).astype(int)
    cyto_features['cyto_gain_count'] = cyto_col.str.count(r'\+')
    cyto_features['cyto_has_gain'] = (cyto_features['cyto_gain_count'] > 0).astype(int)
    cyto_features['cyto_loss_count'] = cyto_col.str.count(r'-[0-9XY]')
    cyto_features['cyto_has_loss'] = (cyto_features['cyto_loss_count'] > 0).astype(int)
    cyto_features['cyto_other_count'] = cyto_col.str.count(r'add\(|ins\(|dup\(')
    cyto_features['cyto_total_anomalies'] = (
        cyto_features['cyto_del_count'] + cyto_features['cyto_transloc_count'] + 
        cyto_features['cyto_inv_count'] + cyto_features['cyto_gain_count'] + 
        cyto_features['cyto_loss_count'] + cyto_features['cyto_other_count']
    )
    cyto_features['cyto_normal'] = cyto_col.str.match(r'^46,(xx|xy)(\[\d+\])?$', case=False).astype(int)
    cyto_features['cyto_complex'] = (
        (cyto_features['cyto_total_anomalies'] >= 3) | 
        cyto_col.str.contains('complex', case=False, na=False)
    ).astype(int)
    chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y']
    for chrom in chromosomes:
        pattern = rf'(\b|[,\(]){chrom}([,;:\)\[]|[pq])'
        cyto_features[f'cyto_chr{chrom}_affected'] = cyto_col.str.contains(
            pattern, case=False, na=False, regex=True
        ).astype(int)
    cyto_features['cyto_monosomy7'] = cyto_col.str.contains(r'-7[^0-9]|^45.*-7', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_trisomy8'] = cyto_col.str.contains(r'\+8[^0-9]|^47.*\+8', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del5q'] = cyto_col.str.contains(r'del\(5\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del20q'] = cyto_col.str.contains(r'del\(20\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr3_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(3[;,:\)]', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr7_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(7[;,:\)]', case=False, na=False, regex=True).astype(int)
    return cyto_features.fillna(0)

def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    mol_features = pd.DataFrame({'ID': patient_ids})
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'), ('vaf_max', 'max'), ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    mol_features = mol_features.merge(effect_counts.reset_index(), on='ID', how='left')
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    return mol_features.set_index('ID')

def create_advanced_features(X_df):
    X_adv = X_df.copy()
    X_adv['blast_to_wbc'] = X_adv['BM_BLAST'] / (X_adv['WBC'] + 1)
    X_adv['monocyte_ratio'] = X_adv['MONOCYTES'] / (X_adv['WBC'] + 1)
    X_adv['anc_ratio'] = X_adv['ANC'] / (X_adv['WBC'] + 1)
    X_adv['platelet_to_blast'] = X_adv['PLT'] / (X_adv['BM_BLAST'] + 1)
    X_adv['hb_to_plt'] = X_adv['HB'] / (X_adv['PLT'] + 1)
    X_adv['vaf_mutation_burden'] = X_adv['vaf_mean'] * X_adv['mutation_count_total']
    X_adv['vaf_per_mutation'] = X_adv['vaf_sum'] / (X_adv['mutation_count_total'] + 1)
    X_adv['cytogenetic_risk_score'] = (
        X_adv['cyto_complex'] * 3 + X_adv['cyto_chr7_affected'] * 2 + 
        X_adv['cyto_del5q'] * 1.5 + X_adv['cyto_loss_count'] * 0.5
    )
    X_adv['blast_cytogenetic_risk'] = X_adv['BM_BLAST'] * X_adv['cytogenetic_risk_score']
    X_adv['blast_to_mutation'] = X_adv['BM_BLAST'] / (X_adv['mutation_count_total'] + 1)
    X_adv['wbc_plt_index'] = X_adv['WBC'] * X_adv['PLT'] / 1000
    X_adv['blast_hb_ratio'] = X_adv['BM_BLAST'] / (X_adv['HB'] + 1)
    X_adv['monocyte_blast_ratio'] = X_adv['MONOCYTES'] / (X_adv['BM_BLAST'] + 1)
    X_adv['mutation_per_vaf'] = X_adv['mutation_count_total'] / (X_adv['vaf_mean'] + 0.01)
    X_adv['cyto_anomaly_density'] = X_adv['cyto_total_anomalies'] / (X_adv['cyto_total_anomalies'].max() + 1)
    X_adv['blast_mutation_interaction'] = X_adv['BM_BLAST'] * X_adv['mutation_count_total']
    X_adv['blast_vaf_interaction'] = X_adv['BM_BLAST'] * X_adv['vaf_mean']
    X_adv['blast_cyto_complex'] = X_adv['BM_BLAST'] * X_adv['cyto_complex']
    X_adv['tp53_blast'] = X_adv['gene_TP53_present'] * X_adv['BM_BLAST']
    X_adv['runx1_mutation_burden'] = X_adv['gene_RUNX1_count'] * X_adv['mutation_count_total']
    X_adv['nras_vaf'] = X_adv['gene_NRAS_present'] * X_adv['vaf_mean']
    X_adv['cyto_mutation_interaction'] = X_adv['cyto_total_anomalies'] * X_adv['mutation_count_total']
    X_adv['chr7_blast'] = X_adv['cyto_chr7_affected'] * X_adv['BM_BLAST']
    X_adv['vaf_cyto_burden'] = X_adv['vaf_sum'] * X_adv['cyto_total_anomalies']
    X_adv['vaf_tp53'] = X_adv['vaf_mean'] * X_adv['gene_TP53_present']
    X_adv['log_wbc'] = np.log1p(X_adv['WBC'])
    X_adv['log_plt'] = np.log1p(X_adv['PLT'])
    X_adv['log_blast'] = np.log1p(X_adv['BM_BLAST'])
    X_adv['log_mutation_count'] = np.log1p(X_adv['mutation_count_total'])
    X_adv['log_vaf_sum'] = np.log1p(X_adv['vaf_sum'])
    return X_adv

# Create features
cyto_features_train = create_cytogenetic_features(clinical_train)
train_patient_ids = clinical_train['ID'].unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids)

target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID').loc[target_clean.index]

numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_numeric = clinical_train_clean[numeric_features].copy()
center_encoded = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)
X_clinical = pd.concat([X_numeric, center_encoded], axis=1)

mol_features_train_aligned = mol_features_train.reindex(X_clinical.index, fill_value=0)
cyto_features_train_aligned = cyto_features_train.reindex(X_clinical.index, fill_value=0)
X_all_base = pd.concat([X_clinical, mol_features_train_aligned, cyto_features_train_aligned], axis=1)
X_all = create_advanced_features(X_all_base)

y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

print(f"✓ Base features created: {X_all.shape[1]} total")

✓ Base features created: 162 total


## 4. Prepare Two Feature Sets

In [15]:
print("="*60)
print("PREPARING TWO OPTIMIZED FEATURE SETS")
print("="*60)

# === RSF: 90 selected features ===
available_rsf = [f for f in RSF_FEATURES if f in X_all.columns]
X_rsf = X_all[available_rsf].copy()

imputer_rsf = SimpleImputer(strategy='median')
X_rsf_imputed = pd.DataFrame(
    imputer_rsf.fit_transform(X_rsf),
    index=X_rsf.index,
    columns=X_rsf.columns
)

print(f"\n✓ RSF features: {len(available_rsf)}")

# === XGBoost: All features → will select best via importance ===
imputer_xgb = SimpleImputer(strategy='median')
X_xgb_imputed = pd.DataFrame(
    imputer_xgb.fit_transform(X_all),
    index=X_all.index,
    columns=X_all.columns
)

print(f"✓ XGBoost baseline: {X_xgb_imputed.shape[1]} features (will be selected)")

# Split (same for both)
X_rsf_train, X_rsf_val, y_train, y_val = train_test_split(
    X_rsf_imputed, y_surv, test_size=0.3, random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

X_xgb_train = X_xgb_imputed.loc[X_rsf_train.index]
X_xgb_val = X_xgb_imputed.loc[X_rsf_val.index]

print(f"\n✓ Split: {len(X_rsf_train)} train, {len(X_rsf_val)} val")
print("="*60)

PREPARING TWO OPTIMIZED FEATURE SETS

✓ RSF features: 90
✓ XGBoost baseline: 162 features (will be selected)

✓ Split: 2221 train, 952 val


## 5. Quick XGBoost for Feature Selection

In [21]:
print("="*60)
print("XGBOOST: FEATURE SELECTION (Like V6.1)")
print("="*60)

# Train quick model to get importance
y_train_xgb = y_train['OS_YEARS'].copy()
y_train_xgb[~y_train['OS_STATUS']] = -y_train_xgb[~y_train['OS_STATUS']]

y_val_xgb = y_val['OS_YEARS'].copy()
y_val_xgb[~y_val['OS_STATUS']] = -y_val_xgb[~y_val['OS_STATUS']]

dtrain_all = xgb.DMatrix(X_xgb_train, label=y_train_xgb)
dval_all = xgb.DMatrix(X_xgb_val, label=y_val_xgb)

model_quick = xgb.train(
    XGB_PARAMS_BASE,
    dtrain_all,
    num_boost_round=100,
    verbose_eval=False
)

# Get feature importance
importance_dict = model_quick.get_score(importance_type='gain')
importance_df = pd.DataFrame([
    {'feature': k, 'importance': v} 
    for k, v in importance_dict.items()
]).sort_values('importance', ascending=False).reset_index(drop=True)

# Calculate cumulative importance
total_importance = importance_df['importance'].sum()
importance_df['cumulative_pct'] = importance_df['importance'].cumsum() / total_importance

# Select features for 90% cumulative importance (like V6.1)
n_selected = (importance_df['cumulative_pct'] <= 0.90).sum()
xgb_selected_features = importance_df.head(n_selected)['feature'].tolist()

print(f"\n✓ Selected {n_selected} features (90% cumulative importance)")
print(f"  Reduction: {X_xgb_train.shape[1]} → {n_selected} features")
print(f"\n  Top 10: {xgb_selected_features[:10]}")

XGBOOST: FEATURE SELECTION (Like V6.1)

✓ Selected 61 features (90% cumulative importance)
  Reduction: 162 → 61 features

  Top 10: ['vaf_cyto_burden', 'blast_cytogenetic_risk', 'chr7_blast', 'blast_mutation_interaction', 'gene_RUNX1_count', 'log_mutation_count', 'cyto_chr5_affected', 'vaf_tp53', 'cyto_mutation_interaction', 'effect_inframe_codon_loss']


## 6. Train Individual Models

In [22]:
print("="*60)
print("MODEL 1: RANDOM SURVIVAL FOREST")
print("="*60)

rsf = RandomSurvivalForest(**RSF_PARAMS)
rsf.fit(X_rsf_train, y_train)

y_pred_rsf_val = rsf.predict(X_rsf_val)

c_index_rsf = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_rsf_val
)[0]

print(f"\nC-index: {c_index_rsf:.4f}")

MODEL 1: RANDOM SURVIVAL FOREST

C-index: 0.7404


In [23]:
print("="*60)
print(f"MODEL 2: XGBOOST OPTIMIZED ({n_selected} features)")
print("="*60)

# Train with selected features
X_xgb_train_sel = X_xgb_train[xgb_selected_features]
X_xgb_val_sel = X_xgb_val[xgb_selected_features]

dtrain_sel = xgb.DMatrix(X_xgb_train_sel, label=y_train_xgb)
dval_sel = xgb.DMatrix(X_xgb_val_sel, label=y_val_xgb)

xgb_model = xgb.train(
    XGB_PARAMS_BASE,
    dtrain_sel,
    num_boost_round=500,
    evals=[(dval_sel, 'val')],
    early_stopping_rounds=50,
    verbose_eval=False
)

y_pred_xgb_val = xgb_model.predict(dval_sel)

c_index_xgb = concordance_index_censored(
    y_val['OS_STATUS'], y_val['OS_YEARS'], y_pred_xgb_val
)[0]

print(f"\nC-index: {c_index_xgb:.4f}")
print(f"Best iteration: {xgb_model.best_iteration}")

MODEL 2: XGBOOST OPTIMIZED (61 features)

C-index: 0.7401
Best iteration: 55


## 7. Optimize Ensemble Weights

In [24]:
print("="*60)
print("OPTIMIZING ENSEMBLE WEIGHTS")
print("="*60)

def objective(trial):
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_rsf = 1 - w_xgb
    
    y_pred_ensemble = w_xgb * y_pred_xgb_val + w_rsf * y_pred_rsf_val
    
    c_index = concordance_index_censored(
        y_val['OS_STATUS'], 
        y_val['OS_YEARS'], 
        y_pred_ensemble
    )[0]
    
    return c_index

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_w_xgb = study.best_params['w_xgb']
best_w_rsf = 1 - best_w_xgb
best_c_index = study.best_value

print(f"\n✓ Optimization complete")
print(f"\nBest weights:")
print(f"  XGBoost: {best_w_xgb:.3f}")
print(f"  RSF:     {best_w_rsf:.3f}")
print(f"\nBest C-index: {best_c_index:.4f}")

OPTIMIZING ENSEMBLE WEIGHTS


  0%|          | 0/100 [00:00<?, ?it/s]


✓ Optimization complete

Best weights:
  XGBoost: 0.996
  RSF:     0.004

Best C-index: 0.7427


## 8. Performance Comparison

In [ ]:
print("="*60)
print("PERFORMANCE COMPARISON")
print("="*60)

results = pd.DataFrame([
    {'Model': 'V4 RSF Baseline', 'Features': 90, 'C-index': 0.7404},
    {'Model': 'V6.1 XGB Optimized', 'Features': n_selected, 'C-index': 0.741},
    {'Model': 'V8.1 RSF', 'Features': len(available_rsf), 'C-index': c_index_rsf},
    {'Model': 'V8.1 XGBoost', 'Features': n_selected, 'C-index': c_index_xgb},
    {'Model': 'V8.1 Ensemble', 'Features': 'Mixed', 'C-index': best_c_index}
])

print("\n" + results.to_string(index=False))

improvement = best_c_index - 0.7404
improvement_vs_best = best_c_index - max(c_index_rsf, c_index_xgb, 0.741)

print(f"\n📊 Improvements:")
print(f"  vs V4 Baseline:    {improvement:+.4f}")
print(f"  vs Best single:    {improvement_vs_best:+.4f}")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(14, 6))

models = ['V4\nRSF', 'V6.1\nXGBoost', 'V8.1\nRSF', 'V8.1\nXGBoost', 'V8.1\nEnsemble']
scores = [0.7404, 0.741, c_index_rsf, c_index_xgb, best_c_index]
colors = ['#808080', '#A23B72', '#2E86AB', '#F18F01', '#27AE60']

bars = ax.bar(models, scores, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

for bar, score in zip(bars, scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{score:.4f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.axhline(y=0.74, color='orange', linestyle='--', linewidth=2, alpha=0.5, label='Baseline')
ax.axhline(y=0.75, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Target: 0.75')
ax.axhline(y=0.76, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Goal: 0.76')

ax.set_ylabel('C-index (Validation)', fontsize=12)
ax.set_title('V8.1 Optimized Ensemble vs Previous Versions', fontsize=14, fontweight='bold')
ax.set_ylim([0.72, max(0.77, best_c_index + 0.01)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Test Set & Submission

In [ ]:
print("="*60)
print("PREPARING TEST SET")
print("="*60)

# Create features for test
cyto_features_test = create_cytogenetic_features(clinical_test)
test_patient_ids = clinical_test['ID'].unique()
mol_features_test = create_molecular_features(molecular_test, test_patient_ids)

clinical_test_indexed = clinical_test.set_index('ID')
X_numeric_test = clinical_test_indexed[numeric_features].copy()
center_encoded_test = pd.get_dummies(clinical_test_indexed['CENTER'], prefix='CENTER', drop_first=True)
for col in center_encoded.columns:
    if col not in center_encoded_test.columns:
        center_encoded_test[col] = 0
center_encoded_test = center_encoded_test[center_encoded.columns]
X_clinical_test = pd.concat([X_numeric_test, center_encoded_test], axis=1)

mol_features_test_aligned = mol_features_test.reindex(X_clinical_test.index, fill_value=0)
cyto_features_test_aligned = cyto_features_test.reindex(X_clinical_test.index, fill_value=0)
X_test_all_base = pd.concat([X_clinical_test, mol_features_test_aligned, cyto_features_test_aligned], axis=1)
X_test_all = create_advanced_features(X_test_all_base)

# RSF test
for feature in available_rsf:
    if feature not in X_test_all.columns:
        X_test_all[feature] = 0
X_test_rsf = X_test_all[available_rsf].copy()
X_test_rsf_imputed = pd.DataFrame(
    imputer_rsf.transform(X_test_rsf),
    index=X_test_rsf.index,
    columns=X_test_rsf.columns
)

# XGBoost test
for col in X_xgb_imputed.columns:
    if col not in X_test_all.columns:
        X_test_all[col] = 0
X_test_all = X_test_all[X_xgb_imputed.columns]
X_test_xgb_imputed = pd.DataFrame(
    imputer_xgb.transform(X_test_all),
    index=X_test_all.index,
    columns=X_test_all.columns
)
X_test_xgb_sel = X_test_xgb_imputed[xgb_selected_features]

print(f"✓ RSF test: {X_test_rsf_imputed.shape}")
print(f"✓ XGBoost test: {X_test_xgb_sel.shape}")

In [ ]:
print("\nRetraining models on full dataset...")

# RSF
rsf_final = RandomSurvivalForest(**RSF_PARAMS)
rsf_final.fit(X_rsf_imputed, y_surv)
print("✓ RSF trained")

# XGBoost
y_full_xgb = target_clean['OS_YEARS'].copy()
y_full_xgb[~target_clean['OS_STATUS']] = -y_full_xgb[~target_clean['OS_STATUS']]
X_xgb_full_sel = X_xgb_imputed[xgb_selected_features]
dfull = xgb.DMatrix(X_xgb_full_sel, label=y_full_xgb)

xgb_final = xgb.train(
    XGB_PARAMS_BASE,
    dfull,
    num_boost_round=xgb_model.best_iteration,
    verbose_eval=False
)
print("✓ XGBoost trained")
print("\n✓ All models retrained")

In [ ]:
# Generate ensemble predictions
dtest = xgb.DMatrix(X_test_xgb_sel)

y_pred_rsf_test = rsf_final.predict(X_test_rsf_imputed)
y_pred_xgb_test = xgb_final.predict(dtest)

# Ensemble with optimized weights
y_pred_ensemble_test = best_w_xgb * y_pred_xgb_test + best_w_rsf * y_pred_rsf_test

# Convert to risk scores
min_pred = y_pred_ensemble_test.min()
max_pred = y_pred_ensemble_test.max()
risk_scores = 1 - (y_pred_ensemble_test - min_pred) / (max_pred - min_pred)

submission = pd.DataFrame({
    'ID': X_test_rsf_imputed.index,
    'risk_score': risk_scores
})

submission_path = f"{DATA_PATH}\\submission_v8.1_optimized_ensemble.csv"
submission.to_csv(submission_path, index=False)

print("="*60)
print("SUBMISSION GENERATED")
print("="*60)
print(f"File: {submission_path}")
print(f"Predictions: {len(submission)}")
print(f"\n📊 Risk Scores (0-1):")
print(submission['risk_score'].describe())

## 10. Summary

In [ ]:
print("="*60)
print("VERSION 8.1 - OPTIMIZED ENSEMBLE SUMMARY")
print("="*60)

print("\n🎯 ENSEMBLE STRATEGY:")
print(f"  RSF: {len(available_rsf)} features (hand-selected from V4)")
print(f"  XGBoost: {n_selected} features (automatic selection, 90% importance)")
print(f"  Weights: XGBoost={best_w_xgb:.3f}, RSF={best_w_rsf:.3f}")

print("\n📊 INDIVIDUAL MODELS:")
print(f"  RSF:     {c_index_rsf:.4f}")
print(f"  XGBoost: {c_index_xgb:.4f}")

print("\n🏆 ENSEMBLE:")
print(f"  C-index: {best_c_index:.4f}")

print("\n📈 IMPROVEMENT:")
print(f"  vs V4:          {improvement:+.4f}")
print(f"  vs Best single: {improvement_vs_best:+.4f}")

if best_c_index >= 0.76:
    print("\n🏆 GOAL EXCEEDED! (C-index ≥ 0.76)")
elif best_c_index >= 0.75:
    print("\n🎯 TARGET REACHED! (C-index ≥ 0.75)")
elif best_c_index > 0.74:
    print("\n✅ IMPROVED vs Baseline!")

print("\n✅ OUTPUT:")
print(f"  Submission: submission_v8.1_optimized_ensemble.csv")
print(f"  Ready for submission!")